In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from lightgbm import LGBMClassifier
from sqlalchemy import create_engine
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, classification_report

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

query = """
SELECT *
FROM hpce.churn_model_dataset
WHERE feature_date <= '2018-06-05'
"""

df = pd.read_sql(query, engine)
df["feature_date"] = pd.to_datetime(df["feature_date"]).dt.date

print(df.shape)
print(df["feature_date"].min(), df["feature_date"].max())

(313289, 26)
2016-09-04 2018-06-05


In [3]:
FEATURES = [
    "daily_orders",
    "daily_spend",
    "f_90",
    "m_90",
    "f_trend_90",
    "m_trend_90",
    "recency_days"
]

cutoff = pd.to_datetime("2018-03-01").date()

train = df[df["feature_date"] <= cutoff].copy()
test  = df[df["feature_date"] > cutoff].copy()

X_train = train[FEATURES]
y_train = train["churn_90d"].astype(int)

X_test  = test[FEATURES]
y_test  = test["churn_90d"].astype(int)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Test base rate:", y_test.mean())

Train: (238067, 7) Test: (75222, 7)
Test base rate: 0.4541889340884316


In [4]:
def show_class_balance(y, name):
    y = pd.Series(y)
    counts = y.value_counts().sort_index()
    rate = y.mean()
    print(f"\n{name} size:", len(y))
    print(f"{name} churn rate (mean of y): {rate:.4f}")
    print(f"{name} counts:")
    print(counts)

show_class_balance(y_train, "TRAIN")
show_class_balance(y_test, "TEST")


TRAIN size: 238067
TRAIN churn rate (mean of y): 0.6942
TRAIN counts:
churn_90d
0     72795
1    165272
Name: count, dtype: int64

TEST size: 75222
TEST churn rate (mean of y): 0.4542
TEST counts:
churn_90d
0    41057
1    34165
Name: count, dtype: int64


In [5]:
lgbm = LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 165272, number of negative: 72795
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001397 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1046
[LightGBM] [Info] Number of data points in the train set: 238067, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,'binary'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [6]:
pred = lgbm.predict_proba(X_test)[:, 1]

roc = roc_auc_score(y_test, pred)
pr  = average_precision_score(y_test, pred)

print(f"LGBM ROC-AUC: {roc:.6f}")
print(f"LGBM PR-AUC : {pr:.6f}")

LGBM ROC-AUC: 0.811457
LGBM PR-AUC : 0.847230


In [7]:
threshold = 0.50
y_pred = (pred >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Threshold:", threshold)
print("Confusion Matrix [ [TN FP], [FN TP] ]:\n", cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

Threshold: 0.5
Confusion Matrix [ [TN FP], [FN TP] ]:
 [[33231  7826]
 [10423 23742]]

Classification report:
              precision    recall  f1-score   support

           0     0.7612    0.8094    0.7846     41057
           1     0.7521    0.6949    0.7224     34165

    accuracy                         0.7574     75222
   macro avg     0.7567    0.7522    0.7535     75222
weighted avg     0.7571    0.7574    0.7563     75222



In [8]:
def recall_at_top_k(y_true, y_prob, k=0.10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    k_n = int(np.ceil(k * n))
    idx = np.argsort(-y_prob)[:k_n]  # highest-risk customers
    return y_true[idx].sum() / max(y_true.sum(), 1)

for k in [0.05, 0.10, 0.20]:
    print(f"Recall@Top{int(k*100)}%:", round(recall_at_top_k(y_test, pred, k), 4))

Recall@Top5%: 0.1086
Recall@Top10%: 0.2174
Recall@Top20%: 0.4345
